Import Dependencies

In [2]:
import openai
import instructor
from pydantic import BaseModel,Field
from qdrant_client import QdrantClient

In [2]:
from dotenv import load_dotenv
load_dotenv("../../.env")

True

Mock Example

In [3]:
prompt="""you are an intelligent assistant.
return asnwer to the question
Question: What is your name?"""

In [4]:
response=openai.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[
        {"role":"system","content":prompt}
        ],
    reasoning_effort="none"

)

print(response.choices[0].message.content)

My name is **OpenAI**.


In [5]:
response

ChatCompletion(id='chatcmpl-E0XGKdbC9m5pJDrfwkyfGNE8bLsFp', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='My name is ChatGPT.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1783796568, model='gpt-5.4-2026-03-05', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=9, prompt_tokens=27, total_tokens=36, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

Add instructor (formatted output)

In [5]:
client=instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [6]:
class Answer(BaseModel):
    answer:str=Field( description="Answer to the question")

In [11]:
response=client.create(
    messages=[
        {"role":"system","content":prompt}
    ],
    reasoning={"effort":"none"},
    response_model=Answer
)

In [14]:
response,raw_response=client.create_with_completion(
    messages=[
        {"role":"system","content":prompt}
    ],
    reasoning={"effort":"none"},
    response_model=Answer
)

In [15]:
response

Answer(answer='I’m ChatGPT.')

In [16]:
raw_response

Response(id='resp_0cd98d346ae2414d006a52a54429c8819f885688652fdf6966', created_at=1783801156.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"answer":"I’m ChatGPT."}', call_id='call_47Lj9SN9BYWAlXi1w5pZPgGT', name='Answer', type='function_call', id='fc_0cd98d346ae2414d006a52a544bdd4819fb8304b64833e3e73', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice=ToolChoiceFunction(name='Answer', type='function'), tools=[FunctionTool(name='Answer', parameters={'properties': {'answer': {'description': 'Answer to the question', 'title': 'Answer', 'type': 'string'}}, 'required': ['answer'], 'title': 'Answer', 'type': 'object', 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Correctly extracted `Answer` with all the required parameters with correct types', output_schema=None)], top_p=0.98, ba

In [17]:
class AnswerWithReasoning(BaseModel):
    reasoning:str=Field(description="Reasoning for the answer")
    answer:str=Field( description="Answer to the question")

In [18]:
response,raw_response=client.create_with_completion(
    messages=[
        {"role":"system","content":prompt}
    ],
    reasoning={"effort":"none"},
    response_model=AnswerWithReasoning
)

In [19]:
response

AnswerWithReasoning(reasoning='The user asks for my name; as an AI assistant, I can identify myself as ChatGPT.', answer='You can call me ChatGPT.')

In [20]:
raw_response

Response(id='resp_0f28916d1f19f61c006a52a8858a1881919cc87dff850de22a', created_at=1783801989.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-nano-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"reasoning":"The user asks for my name; as an AI assistant, I can identify myself as ChatGPT.","answer":"You can call me ChatGPT."}', call_id='call_FQcTVq9zBC9iRwq775NogVSe', name='AnswerWithReasoning', type='function_call', id='fc_0f28916d1f19f61c006a52a88665288191aea2b6dc85fd78e6', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice=ToolChoiceFunction(name='AnswerWithReasoning', type='function'), tools=[FunctionTool(name='AnswerWithReasoning', parameters={'properties': {'reasoning': {'description': 'Reasoning for the answer', 'title': 'Reasoning', 'type': 'string'}, 'answer': {'description': 'Answer to the question', 'title': 'Answer', 'type': 'string'}}, 'required': ['reasoning', 'answ

RAG Pipeline

In [24]:
class RAG_GenerationResponse(BaseModel):
    answer:str=Field(description="Answer to the question")

In [25]:
qdrant_client = QdrantClient(url="http://localhost:6333")


def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
   
    return response.data[0].embedding   


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt




def generate_answer(prompt):

    response,raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning={"effort":"none"},
        response_model=RAG_GenerationResponse
    )
    
    return response


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer= {

        "answer":answer.answer,
        "question":question,
        "retrieved_context_ids":retrieved_context["retrieved_context_ids"],
        "retrieved_context":retrieved_context["retrieved_context"]
    }
    return final_answer

def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

In [26]:
output=rag_pipeline("any earbud")

In [27]:
output

{'answer': 'Yes—here are some earbuds/headphones available:\n1) Jesebang Wireless Earbud (Black) — Bluetooth 5.3 stereo, up to 40H, ENC mic, USB‑C, LED display, IP7 waterproof, sport earhooks.\n2) LUDOS Turbo Wired Earbuds — wired in‑ear ergonomic design with microphone, memory foam, durable cable, bass.\n3) Earbay Bluetooth Headset with Microphone (Rose Gold) — on‑ear headset, ENC noise-cancelling mic, USB dongle, mute button, up to 26hrs talk time, multipoint.\n4) 2-Pack Apple Wired Earbuds (Lightning) — wired earbuds with built-in mic and volume control, compatible with iPhone models with Lightning connector.\n5) Earbuds Cleaning Pen / Earbuds Cleaning Kit (Green) — 3-in-1 cleaning kit with soft brush, flocking sponge, and metal pen tip.',
 'question': 'any earbud',
 'retrieved_context_ids': ['B0C2TLBSMP',
  'B09XB59YPR',
  'B07VSWK14G',
  'B0C6KM4H9M',
  'B0CCK3KL19'],
 'retrieved_context': ['Jesebang Wireless Earbud, Bluetooth Headphones 5.3 Stereo Earphones 2023 Ear Buds 40H ENC 